Projeto: ***Predição de Diabetes: Comparação entre Baseline Probabilístico e Regressão Logística***

In [1]:
!wget "https://raw.githubusercontent.com/arthur-202/Portfolio/main/car_price_prediction/diabetes%20(1).csv"

--2026-01-02 21:05:31--  https://raw.githubusercontent.com/arthur-202/Portfolio/main/car_price_prediction/diabetes%20(1).csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23875 (23K) [text/plain]
Saving to: ‘diabetes (1).csv’

diabetes (1).csv    100%[===================>]  23.32K  --.-KB/s    in 0.006s  

2026-01-02 21:05:37 (4.05 MB/s) - ‘diabetes (1).csv’ saved [23875/23875]



In [2]:
!mv diabetes\ \(1\).csv diabetes.csv

In [3]:
# Etapa 1: Leitura e armazenamento do dataset
import pandas as pd
import numpy as np
dados = pd.read_csv('diabetes.csv')

In [4]:
# Verificando se existem dados faltantes
round((dados.isnull().sum() / dados.shape[0]) * 100, 2)

Pregnancies                 0.0
Glucose                     0.0
BloodPressure               0.0
SkinThickness               0.0
Insulin                     0.0
BMI                         0.0
DiabetesPedigreeFunction    0.0
Age                         0.0
Outcome                     0.0
dtype: float64

In [5]:
# Separando dados de treino em input e target
X = dados.drop('Outcome', axis=1) # input
Y = dados['Outcome'] # Target


In [6]:
# Normalizando os valores das features com MinMaxScale
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))
num_col = [col for col in X.columns if X[col].dtype != 'object']
x1 = X
x1[num_col] = scaler.fit_transform(x1[num_col])

x1.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0.352941,0.743719,0.590164,0.353535,0.000000,0.500745,0.234415,0.483333
1,0.058824,0.427136,0.540984,0.292929,0.000000,0.396423,0.116567,0.166667
2,0.470588,0.919598,0.524590,0.000000,0.000000,0.347243,0.253629,0.183333
3,0.058824,0.447236,0.540984,0.232323,0.111111,0.418778,0.038002,0.000000
4,0.000000,0.688442,0.327869,0.353535,0.198582,0.642325,0.943638,0.200000


**Análise de Correlação entre os Dados**

In [7]:
# Análise de correlação entre os dados
import seaborn as sns
import matplotlib.pyplot as plt

corr = dados.corr()
plt.figure(dpi=130)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.show()

print(corr['Outcome'].sort_values(ascending=False))

ModuleNotFoundError: No module named 'seaborn'

Observações: *Nota-se uma alta correlação entre os valores da glicose com a diabetes - pois, quanto maior a glicose, maior é a chance de está diabético*

In [ ]:
# Separando os dados x e y de treinamento com os de teste
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x1, Y, test_size=0.3, random_state=1)

In [ ]:
# Etapa de treinamento do modelo
from sklearn.linear_model import LogisticRegression
train = LogisticRegression(max_iter=10000, random_state=1)
train.fit(X_train, y_train)

LogisticRegression(max_iter=10000, random_state=0)

In [ ]:
# Avaliando as métricas de acurácia
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, train.predict(X_test)) * 100
print(f"Logistic Regression model accuracy: {acc:.2f}%")

Logistic Regression model accuracy: 77.92%


In [ ]:
# Matriz de confusão
from sklearn.metrics import confusion_matrix
y_pred = train.predict(X_test)
confusion_matrix(y_test, y_pred)

array([[134,  12],
       [ 39,  46]])

# **Métrica Probabilística**
**A partir de agora, será analisada com base da métrica Log Loss.**
A métrica Log Loss avalia a qualidade das probabilidades estimadas pelo modelo, penalizando previsões confiantes e erradas.

*Diferentemente da acurácia, ela considera toda a distribuição
𝑃( 𝑌 ∣ X )*

In [ ]:
# A biblioteca sklearn possui o módulo log_loss
from sklearn.metrics import log_loss
y_probabilidade = train.predict_proba(X_test)[:, 1]


In [ ]:
# Cálculo de Log_loss
lg_lss_modelo = log_loss(y_test, y_probabilidade)
print(f"{lg_lss_modelo:.4f}")

0.4807


In [ ]:
# calcular a probabilidade da classe positiva no treino
p_base = y_train.mean()
# criação de um vetor de probabilidades constantes
y_prob_base = np.full(shape=len(y_test), fill_value=p_base)

In [ ]:
# Log loss da baseline
ll_baseline = log_loss(y_test, y_prob_base)
print(f"Log Loss - Baseline probabilístico: {ll_baseline:.4f}")
print(f"Log Loss - Modelo treinado: {lg_lss_modelo:.4f}")

Log Loss - Baseline probabilístico: 0.6595
Log Loss - Modelo treinado: 0.4807
